In [ ]:

from operator import add
from typing import Annotated

from langchain.chat_models import init_chat_model
from langgraph.graph import END, START, StateGraph
from langgraph.types import Send
from typing_extensions import TypedDict

llm = init_chat_model("openai:gpt-4o-mini")

In [ ]:
class State(TypedDict):
    user_query: str
    problem: str
    loop_count: int
    solution: str
    retry_reason: Annotated[list[str], add]

In [ ]:
def problem_maker(state: State):
    
    response = llm.invoke(
        f"""
        당신은 수학문제 생성 전문가입니다. 
        사용자가 특정 개념을 입력하면 그 개념을 제대로 이해하고 있는지 확인할 수 있는
        문제를 생성해야 합니다.
        
        사용자가 입력한 개념: {state.get("user_query")}
        
        재시도 이유가 있는 경우 처음 생성하는게 아니기 떄문에
        재시도 이유를 분석해서 문제를 다시 생성해야 합니다.
        {state["retry_reason"]}
        """
    )
    
    return {
        "problem": response.content,
        "loop_count": state.get("loop_count", 0) + 1
    }

def problem_solver(state: State):
    pass

def problem_output(state: State):
    pass


In [ ]:
graph_builder = StateGraph(State)

graph_builder.add_node("problem_maker", problem_maker)
graph_builder.add_node("problem_solver", problem_solver)
graph_builder.add_node("problem_output", problem_output)

graph_builder.add_edge(START, "problem_maker")
graph_builder.add_edge("problem_maker", "problem_solver")
graph_builder.add_conditional_edges("problem_solver", {
    True: "problem_output",
    False: "problem_maker"
})

graph = graph_builder.compile()
graph

In [ ]:
with open("fed_transcript.md", "r", encoding="utf-8") as file:
    document = file.read()


for chunk in graph.stream(
    {"document": document},
    stream_mode="updates",
):
    print(chunk, "\n")